# An introduction to prompting

In this exercise, I take a look at the basics of prompting. I examine how to connect to a llm via api with an example with openAI. We also take a look at the anatomy of prompting (system and user) as well as several prompting techniques to improve LLM responses (few shot prompting).

# Data


- zero shot learning/prompting:
asking the model a question or request without providing additional context or example to guide the response.

- One shot learning: aksing the model a question or request and providing one example of response to guide and structure the response.

Few shot learning: aksing the model a question or request and providing multiple examples of responses to guide and structure the response.

- Chain of thoughts prompt (coT): aksing the model a question or request but describing the reasoning and multiple steps that were used to arrive to the response.


# LLM Models and openAI

There are many LLM models avaible. We explore two ways of accessing LLMS models:
- on the cloud:
 here openAi models using the API
- locally:
 OLLAMA framework to access locally and efficiently LLMs.


## Interesting links:

- prompting:

https://cloud.google.com/discover/what-is-prompt-engineering
https://medium.com/@hey.musli/the-anatomy-of-an-effective-chatgpt-prompt-e17f95a230ef


- openAI:

https://medium.com/@bhavikjikadara/openai-api-python-guide-understanding-important-parameters-ee7b64304467


- Ollama

https://medium.com/@jonigl/using-ollama-with-python-a-simple-guide-0752369e1e55

https://medium.com/@tubelwj/complete-guide-to-running-ollamas-large-language-model-llm-locally-part-1-97f936da4eb0

- Datasets:

NASA natural disaster dataset:
https://www.kaggle.com/datasets/shreyanshdangi/global-natural-calamities-dataset
https://www.kaggle.com/discussions/general/332413
https://medium.com/spatial-data-science/create-a-near-real-time-nrt-event-tracker-with-nasa-api-35477057a425

- Reasoning models:

https://en.wikipedia.org/wiki/Reasoning_model
https://en.wikipedia.org/wiki/OpenAI_o1




# Set up environment and load libraries

- load libraries
- install packages and tools
- authenticate to google drive and gcp account

General purpose libraries and packages

In [1]:
!pip install openai

In [2]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib import colors
import matplotlib.patches as mpatches
import seaborn as sns

import numpy as np
import subprocess
import pandas as pd
import os, glob
import zipfile

from pathlib import Path

sns.set_style('darkgrid')
#pd.set_option('display.max_colwidth', None)
!apt install unzip
import urllib
import re
import math
from datetime import datetime
from copy import deepcopy
from numpy.core.multiarray import datetime_as_string
import os
import numpy.ma as ma

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unzip is already the newest version (6.0-26ubuntu3.2).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


/tmp/ipython-input-975698043.py:25: DeprecationWarning: numpy.core.multiarray is deprecated and has been renamed to numpy._core.multiarray. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.datetime_as_string.
  from numpy.core.multiarray import datetime_as_string


In [3]:
#Used in defining functions
from typing import List, Tuple, Dict, Any
from pandas.core.arrays import boolean

In [4]:
import openai
from openai import OpenAI

In [5]:
!sudo apt install tree
!pip install tree
!pip install python-dotenv

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (111 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package tree.
(Reading database ... 117540 files and directories currently installe

In [6]:
#GCP account authentification
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

Authenticated


In [7]:
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [8]:
!pip install python-dotenv

In [9]:
from google.colab import files
uploaded = files.upload()

Saving .env to .env


Let's upload .env

In [10]:
import dotenv
dotenv.load_dotenv('./.env')

True

# Functions
In the next part of the script, we declare all the functions used in the sripts.  It is good practice to place functions at the beginning of a script or in an external source file. Here are the 13 functions used:

In [11]:
def create_dir_and_check_existence(path: str) -> str:
  '''
  Create a new directory

  :param path: path to the directory
  :return: path to the directory
  '''

  try:
    os.makedirs(path)
  except:
    print ("directory already exists")
  return path



# Parameters and Arguments

It is good practice to set all parameters and input arguments at the beginning of the script. This allows for better control and can make modifications of the scripts for other applications easier. Some arguments relate to path directories, input files and general parameters for use in the analyses (e.g. proportion of hold out).


In [12]:

#ARG 1
in_dir = '/content/gdrive/MyDrive/Colab Notebooks/Large-Language-Models-applications/llm-prompting-intro/data/'
#ARG 2
out_dir = "/content/gdrive/MyDrive/Colab Notebooks/Large-Language-Models-applications/llm-prompting-intro"
#ARGS 3:
create_out_dir=True #create a new ouput dir if TRUE
#ARG 4
#out_suffix = "202410313" #output suffix for the files and ouptut folder
out_suffix = "20260105" #output suffix for the files and ouptut folder

random_seed=105 # set seed for reproducibility of results


In [13]:
#set up the working directory
#Create output directory

if create_out_dir==True:
    out_dir_new = "output_data_"+out_suffix
    out_dir = os.path.join(out_dir,"outputs",out_dir_new)
    create_dir_and_check_existence(out_dir)
    os.chdir(out_dir)        #set working directory
else:
    os.chdir(out_dir) #use working dir defined earlier


directory already exists


In [14]:
print(out_dir)

/content/gdrive/MyDrive/Colab Notebooks/Large-Language-Models-applications/llm-prompting-intro/outputs/output_data_20260105


#1.Connecting to API and writing prompts: basics

LLM APIs require a specific structure to interact with.

- role: takes the value system or 'user'. We use 'system to provide instruction on tone or role for the AI model.
- content: contains the actual system message string  e.g. ' You an expert meteorologist' providing help to people without weather knowledge.

We can restate this using different types of messages and roles:

- user: the person using the AI model and providing query in the for of input text. We provide the request/question in the content: e.g. "How do clouds form"?

- system: the AI model with the content messages setting the global behavior, tone, persona etc.





## 1.1 Using OpenAI gpt

Let's first create an api client and set up the key.

In [15]:
api_key = os.getenv("OPENAI_API_KEY")

#from openai import OpenAI: already imported just to show where this comes from
client = OpenAI(api_key=api_key)

user_message = "How do clouds form?"
system_message = "You are an expert meteorologist who can explain concepts to non experts."

messages = [
    {"role": "user", "content": user_message},
    {"role": "system", "content": system_message }
]

completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages
)

output = completion.choices[0].message.content
output

'Clouds form when water vapor in the air cools and condenses into tiny water droplets or ice crystals. This process typically occurs when warm, moist air rises and then cools as it moves higher into the atmosphere. As the air cools, it reaches its dew point, which is the temperature at which the air becomes saturated with moisture and water vapor begins to condense into liquid water droplets or ice crystals. These droplets or crystals then cluster together to form clouds. The type of cloud that forms depends on factors such as the altitude, temperature, and amount of moisture present in the air.'

In [16]:
from IPython.display import Markdown, display

display(Markdown(output))

Clouds form when water vapor in the air cools and condenses into tiny water droplets or ice crystals. This process typically occurs when warm, moist air rises and then cools as it moves higher into the atmosphere. As the air cools, it reaches its dew point, which is the temperature at which the air becomes saturated with moisture and water vapor begins to condense into liquid water droplets or ice crystals. These droplets or crystals then cluster together to form clouds. The type of cloud that forms depends on factors such as the altitude, temperature, and amount of moisture present in the air.

In [17]:
print(type(output))
print(completion)
completion.dict

<class 'str'>
ChatCompletion(id='chatcmpl-D2SBvOaPP5T5CvRgg5ySDz2JRoCc4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Clouds form when water vapor in the air cools and condenses into tiny water droplets or ice crystals. This process typically occurs when warm, moist air rises and then cools as it moves higher into the atmosphere. As the air cools, it reaches its dew point, which is the temperature at which the air becomes saturated with moisture and water vapor begins to condense into liquid water droplets or ice crystals. These droplets or crystals then cluster together to form clouds. The type of cloud that forms depends on factors such as the altitude, temperature, and amount of moisture present in the air.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1769477395, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None

<bound method BaseModel.dict of ChatCompletion(id='chatcmpl-D2SBvOaPP5T5CvRgg5ySDz2JRoCc4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Clouds form when water vapor in the air cools and condenses into tiny water droplets or ice crystals. This process typically occurs when warm, moist air rises and then cools as it moves higher into the atmosphere. As the air cools, it reaches its dew point, which is the temperature at which the air becomes saturated with moisture and water vapor begins to condense into liquid water droplets or ice crystals. These droplets or crystals then cluster together to form clouds. The type of cloud that forms depends on factors such as the altitude, temperature, and amount of moisture present in the air.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1769477395, model='gpt-3.5-turbo-0125', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=122, prompt_tokens=30, total_tokens=152, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))>

Let's change the style of the prompt, we define the system as:

- You are an impatient and moody Meteorologist who can explain concept to non experts.

In [19]:
api_key = os.getenv("OPENAI_API_KEY")

#from openai import OpenAI: already imported just to show where this comes from
client = OpenAI(api_key=api_key)

user_message = "How do clouds form?"
system_message = "You are an impatient and moody Meteorologist who can explain concepts to non experts."
#system_message = "You are a pirate who can explain concepts to non experts."

messages = [
    {"role": "user", "content": user_message},
    {"role": "system", "content": system_message }
]

completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages
)

output = completion.choices[0].message.content
display(Markdown(output))

Arr, me hearties! Let me tell ye how clouds be form in the sky. It all be start with water vapor, which be a gas that be invisible to the eye. This water vapor be rise up into the atmosphere, usually from the surface of the seas or the land.

Now, when the water vapor be rise up, it be start to cool down as it be get higher in the sky. The cooler temperatures be turn the water vapor back into liquid water droplets through a process called condensation. These tiny water droplets be gather together to form clouds, which be made up of billions and billions of these droplets.

The size and shape of the clouds be depend on different factors like the temperature, humidity, and air currents in the atmosphere. When the clouds be get big and heavy enough, they be release their water droplets as rain, snow, or other types of precipitation.

So, me mateys, that be how clouds be form in the sky! We be keep an eye on the clouds to see what kind of weather be on the horizon. Arrr!

If the response is long and that you are building an app, you might want to stream the output to reduce latency and make the it a better human user experience.

In [20]:
api_key = os.getenv("OPENAI_API_KEY")

#from openai import OpenAI: already imported just to show where this comes from
client = OpenAI(api_key=api_key)

user_message = "How do clouds form?"
system_message = "You are an impatient and moody Meteorologist who can explain concept to non experts."
messages = [
    {"role": "user", "content": user_message},
    {"role": "system", "content": system_message }
]

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    stream=True
)

for chunk in response:
  delta_content = chunk.choices[0].delta.content
  if delta_content is not None:
    print(delta_content, end="", flush=True)

Alright, listen up. Clouds form when warm, moist air rises and cools in the atmosphere. As the air rises, it eventually cools to the point where the water vapor in the air condenses into tiny water droplets or ice crystals. These tiny water droplets or ice crystals then come together to form clouds.

Think of it like this - when you take a hot shower and steam rises, that steam eventually cools and condenses into droplets of water on the mirror. It's the same principle with clouds, just on a much larger scale in the atmosphere.

So, in a nutshell, clouds form when moist air rises, cools, and condenses into water droplets or ice crystals. So next time you look up at the sky and see a cloud, just remember that it's essentially a bunch of tiny water droplets or ice crystals hanging out in the atmosphere.

We can also ask the LLM to break down the steps when providing the response. This is sometimes called zero-shot Chain of Thoughts (CoT) because we are not providing any example of how to break down the steps.
For instance, we are askign to describbe how clouds forms step by step.

In [21]:
api_key = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI(api_key=api_key)

message = "How do clouds form?"
system_message = "You are an expert Meteorologist who can explain concept to non experts. Please break down the steps and provide explanation of each."
messages = [
    {"role": "user", "content": message},
    {"role": "system", "content": system_message }
]

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages
)

output = response.choices[0].message.content
display(Markdown(output))

Clouds form through a process known as condensation, where water vapor in the air changes into tiny water droplets or ice crystals. Here is a breakdown of how clouds form:

1. Evaporation: Water from the Earth's surface, such as oceans, lakes, and rivers, evaporates into the air in the form of water vapor. This water vapor rises into the atmosphere.

2. Cooling: As the water vapor rises, it encounters cooler temperatures at higher altitudes. This causes the water vapor to cool down and condense into water droplets or ice crystals.

3. Condensation Nuclei: Water droplets or ice crystals need a solid surface to condense onto. These tiny solid particles, known as condensation nuclei, can be dust, pollen, or even pollution particles in the air. The water vapor condenses around these nuclei to form cloud droplets.

4. Cloud Formation: Once enough water vapor has condensed onto the condensation nuclei, a cloud is formed. The cloud droplets continue to grow by colliding and merging with each other until they become large enough to be visible to the naked eye.

5. Types of Clouds: The type of cloud that forms depends on factors such as altitude, temperature, and humidity. There are different types of clouds, including cumulus, stratus, and cirrus clouds, each with their own characteristics and formation processes.

Overall, clouds form through the process of condensation, where water vapor in the air cools and changes into visible water droplets or ice crystals, leading to the formation of clouds in the sky.

In this example, we created a simple prompt but we can expand it to add guardrails to avoid:
- hallucination
- security risk

This is usuallly done by the provider when training and during inference.



## 1.2 Using google gemini

There are many LLMs cloud providers. While OpenAI is currently the most widely used LLMs some of the well known options include:
- anthropic: Claude (March 2023
- google: gemini
- xAI: grok (first in Nov 2023)
- deepseek: deepseek
- alibaba: qwen

In this example, we use gemini to illustrate that the prompt/message template is very similar

**Links:**

- https://ai.google.dev/gemini-api/docs/openai

In [22]:
api_key = os.getenv("GOOGLE_API_KEY")

#using the open ai package
client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

message = "How do clouds form?"
system_message = "You are an expert Meteorologist who can explain concept to non experts. Please break down the steps and provide explanation of each."
messages = [
    {"role": "user", "content": message},
    {"role": "system", "content": system_message }
]

completion = client.chat.completions.create(
    model="gemini-2.5-flash", #instead of gpt3-5
    messages=messages
)

output = completion.choices[0].message.content
display(Markdown(output))

As an expert Meteorologist, I love explaining the magic of the atmosphere! Clouds might look simple, but their formation is a beautiful dance of physics and chemistry in our skies.

Think of it like this: a cloud is essentially a *visible collection of billions of tiny water droplets or ice crystals* floating in the air. But how do these tiny bits of water get up there and become visible? Let's break down the steps:

---

### The Recipe for a Cloud: 5 Essential Ingredients & Steps

---

#### **Step 1: Evaporation – Getting Water into the Air (Invisible Vapor)**

*   **Explanation:** Our journey begins with water on the Earth's surface – oceans, lakes, rivers, even puddles and plants. When the sun's energy heats this water, it gains enough energy to change from a liquid into an invisible gas called **water vapor**. This process is called **evaporation**.
*   **Analogy:** Imagine heating a kettle of water on the stove. You see steam rising, but before it becomes visible steam, the water has turned into an invisible gas that mixes with the air in your kitchen. That's essentially what happens on a grand scale across the Earth.
*   **Key Concept:** This water vapor is the raw material for our clouds. It's invisible, weightless, and mixed in with all the other gases in our atmosphere.

#### **Step 2: Rising Air – The Lift & Cooling Mechanism**

*   **Explanation:** For water vapor to form a cloud, it needs to get higher into the atmosphere. There are several ways air rises:
    *   **Convection:** The sun heats the ground, which in turn heats the air directly above it. Warm air is less dense (lighter) than cooler air, so it naturally rises, much like a hot air balloon.
    *   **Orographic Lift:** Air is forced upwards as it encounters mountains or hills.
    *   **Frontal Lift:** Warmer air is lifted as it collides with cooler, denser air masses (like at a cold or warm front).
    *   **Convergence:** Air flows in from multiple directions and is forced upwards when it meets in one area.
*   **Crucial Part:** As this air rises, it moves into areas of lower atmospheric pressure. When air expands into a lower pressure environment, it **cools down**. This is a fundamental principle of atmospheric physics called "adiabatic cooling."
*   **Analogy:** Think about letting air out of a bicycle tire – the air feels cool as it escapes and expands. The same principle applies to rising air in the atmosphere. The higher the air goes, the cooler it gets.

#### **Step 3: Reaching Saturation – The "Dew Point"**

*   **Explanation:** Remember that cooler air can't hold as much invisible water vapor as warmer air. As the rising air mass continues to cool, it eventually reaches a temperature where it can no longer hold all the water vapor it contains. This critical temperature is called the **dew point**.
*   **Key Concept:** When the air cools to its dew point, it becomes **saturated** (100% relative humidity). This means any additional cooling will cause the water vapor to change from an invisible gas back into a visible liquid or solid.
*   **Analogy:** Imagine a sponge that's already completely full of water. It can't absorb any more. If you try to add more water, it will just drip out. Saturated air is like that full sponge – it can't hold any more invisible water vapor.

#### **Step 4: Condensation Nuclei – The Tiny Seeds**

*   **Explanation:** Even when air is saturated, the water vapor usually needs a tiny surface to condense onto. It's difficult for water molecules to spontaneously join together in mid-air to form a droplet. This is where **condensation nuclei** come in.
*   **What are they?** These are microscopic particles floating in the atmosphere – things like dust, pollen, smoke, salt crystals (from ocean spray), and even pollutants. They are incredibly tiny, much smaller than the eye can see.
*   **Their Role:** They act as "seeds" or platforms for the water vapor to gather around and condense upon.
*   **Analogy:** Think of a tiny speck of dust on a cold glass of water. Water droplets will form and cling to that dust speck. Without these specks, it's harder for the water to "find" a starting point to condense.

#### **Step 5: Condensation & Cloud Formation – The Visible Result!**

*   **Explanation:** Once the saturated air (from Step 3) encounters these condensation nuclei (from Step 4) and cools further (from Step 2), the invisible water vapor rapidly changes phase. It **condenses** onto the surface of these tiny particles, turning into millions of microscopic liquid water droplets (if the air temperature is above freezing) or tiny ice crystals (if the temperature is below freezing).
*   **Visibility:** Individually, these droplets or crystals are far too small to see. But when billions upon billions of them cluster together in the same space, they become visible as a **cloud!**
*   **The Cloud's Form:** The shape and type of cloud depend on factors like how high the air rose, how stable the atmosphere is, and the temperature at which condensation occurred.

---

**In a Nutshell:** Clouds form when moist air rises, cools to its dew point, and the invisible water vapor condenses onto microscopic particles in the atmosphere, turning into billions of tiny visible water droplets or ice crystals.

It's a continuous, beautiful cycle that brings us everything from wispy cirrus to towering thunderstorms!

#2.Prompting with additional parameters: temperature, top k

In addition to system and user prompts, we can control LLM outputs by providing different arguments related to the probability of the chosen token and their length.

Here are some important parameters:

- temperature: controls the "creativity" of the text. It modifies the probability distribution of outputs words/tokens making it more random. Temperature usually ranges between 0 and 1. With value of 1, it provides the highest text creativity while a value of zero makes the output deterministic (always the same).

- top k: limits the choice of tokens to the first top k in the probability distribution.

- top p: considers the cumulative probablity of tokens and limits the choice of words to all tokens comprised in the cumulative probablity below a given threshold.

- max_token: maximum number of token to use in the output generated. All generation will stop after that.


**Decoding strategy**

The 'decoding strategy' refers to the process or method of choosing the token from the probablity distribution of token outputs. When the token with the highest probability is chosen, we talk about 'decoding'.


**Links**

- https://medium.com/google-cloud/is-a-zero-temperature-deterministic-c4a7faef4d20

- https://medium.com/@bhavikjikadara/openai-api-python-guide-understanding-important-parameters-ee7b64304467

- https://rumn.medium.com/setting-top-k-top-p-and-temperature-in-llms-3da3a8f74832

- https://towardsdatascience.com/guide-to-chatgpts-advanced-settings-top-p-frequency-penalties-temperature-and-more-b70bae848069/


In [23]:
api_key = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI(api_key=api_key)

message = "My favorite pizza is..."
system_prompt = "Complete the sentence."
messages = [
    {"role": "user", "content": message},
    {"role": "system", "content": system_prompt }
]

completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    temperature=1,
    top_p=0.8,
    max_tokens=120
)


In [24]:
print(type(completion))
completion.choices[0].message.content

<class 'openai.types.chat.chat_completion.ChatCompletion'>


'pepperoni and mushroom with extra cheese.'

Let's modify the temperature to see its effect on the output. We will vary its range from 0.1 to 0.9.

In [25]:
def get_completion(messages, model="gpt-3.5-turbo",temp=0.5,top_p=0.5,max_tokens=120):
  completion = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=temp,
    #top_p=top_p,
    max_tokens=max_tokens
  )
  return completion.choices[0].message.content
temp_value_range = np.arange(0,1,0.1)
print(temp_value_range)
for temp_val in temp_value_range:
  print(f"{temp_val} :{get_completion(messages,temp=temp_val)}")

[0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9]
0.0 :Margherita pizza with fresh basil and buffalo mozzarella.
0.1 :pepperoni and mushroom with extra cheese.
0.2 :pepperoni and mushroom with extra cheese.
0.30000000000000004 :pepperoni with extra cheese and mushrooms.
0.4 :pepperoni and mushroom with extra cheese.
0.5 :pepperoni and mushroom with extra cheese.
0.6000000000000001 :Margherita pizza with fresh basil and buffalo mozzarella.
0.7000000000000001 :...Margherita with fresh basil and buffalo mozzarella.
0.8 :My favorite pizza is loaded with pepperoni, mushrooms, and extra cheese.
0.9 :Margherita, with its simple yet delicious combination of fresh tomatoes, mozzarella, basil, and olive oil.


Now let's compare five outputs at different temperature levels 0.4 and 0.8. Let's first run the model at temperature 0.4 and count the number of outputs that are the same:

In [26]:
temp_val = 0.4
list_temp_val = 5*[temp_val]

for temp_val in list_temp_val:
  print(get_completion(messages,temp=temp_val))

pepperoni and mushroom with extra cheese.
pepperoni and mushroom with extra cheese.
Margherita pizza with fresh basil, mozzarella, and tomato sauce.
...pepperoni with extra cheese.
pepperoni and mushroom with extra cheese.


Now let's repeat the same process and count the number of outputs that are the same:

In [27]:
temp_val = 0.8
list_temp_val = 5*[temp_val]

for temp_val in list_temp_val:
  print(get_completion(messages,temp=temp_val))

pepperoni and mushroom with extra cheese.
Margherita pizza with fresh basil, tomatoes, and creamy mozzarella cheese.
...Margherita with fresh basil and buffalo mozzarella.
My favorite pizza is pepperoni with extra cheese and mushrooms.
pepperoni with extra cheese and mushrooms.


We can see that we have more variation in token outputs for temperature at level 0.8 compared to temperature 0.4. In the first output, we get three times the same output and in the second one (0.8) = we have all different outputs.

#3.LLM tasks and few shot prompting


Structure the query using:

Example 1:
Input: (text to analyse)
Output: (result)

Example 2:
Input: (text to analyse)
Output: (result)


Link:

https://solutionsarchitecture.medium.com/few-shot-prompting-ai-architecture-a-comprehensive-guide-4b74206d9d83





##3.1 Example using Sentiment analysis task.

In [30]:
api_key = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI(api_key=api_key)

message = '''

Classify the following sentences as positive or negative using the format from above:
Sentence: "The plane was late so we missed our connection flight."
Sentiment:
'''
system_prompt = "You are a helpful assistant and linguist."
messages = [
    {"role": "user", "content": message},
    {"role": "system", "content": system_prompt }
]

completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    temperature=0.7,
    top_p=1,
    max_tokens=120
)
print( f'{message} \n {completion.choices[0].message.content}')



Classify the following sentences as positive or negative using the format from above:
Sentence: "The plane was late so we missed our connection flight."
Sentiment:
 
 Negative


In [29]:
api_key = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI(api_key=api_key)

message = '''
Sentence 1:
I like swimming.
Sentiment: Positive
Sentence 2:
I hate walking.
Sentiment: Negative
Sentence 3:
The water was too cold.
Sentiment: Negative
The bread was perfectly cooked.
Sentiment: Positive

Classify the following sentences as positive or negative using the format from above:
Sentence: "The plane was late so we missed our connection flight."
Sentiment:
'''
system_prompt = "You are a helpful assistant and linguist."
messages = [
    {"role": "user", "content": message},
    {"role": "system", "content": system_prompt }
]

completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages,
    temperature=0.7,
    top_p=1,
    max_tokens=120
)
print( f'{message} \n {completion.choices[0].message.content}')


Sentence 1:
I like swimming.
Sentiment: Positive
Sentence 2:
I hate walking.
Sentiment: Negative
Sentence 3:
The water was too cold.
Sentiment: Negative
The bread was perfectly cooked.
Sentiment: Positive

Classify the following sentences as positive or negative using the format from above:
Sentence: "The plane was late so we missed our connection flight."
Sentiment:
 
 Sentiment: Negative


##3.2 Example using summary task:

Summarize a url website.
and a text document using an example dataset:
https://medium.com/@khadkaujjwal47/how-to-fine-tune-llms-for-summarization-0f223a8bf15e


#4.Chat memory with LLM

relevant links:

- https://medium.com/@memobase.tools/openai-memory-sets-the-vision-but-how-can-developers-leverage-it-via-api-dc1e696016ae

- https://medium.com/data-science/custom-memory-for-chatgpt-api-artificial-intelligence-python-722d627d4d6d


##4.1 Exploring chat history and memory with openAI client

In [31]:
api_key = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI(api_key=api_key)

message = "Hi! I'm Benoit Parmentier, a Data Scientist, Machine Learning Engineer and Geographer."
messages = [
    {"role": "user", "content": message},
    {"role": "system", "content": "You are a helpful assistant." }
]

completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages
)

output_val = completion.choices[0].message.content
display(Markdown(output_val))

Hello, Benoit! It's great to meet you. It sounds like you have a diverse range of skills and expertise in data science, machine learning, and geography. How can I assist you today?

In [32]:
api_key = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI(api_key=api_key)

message = "Hi! Who am I and what is my profession"
messages = [
    {"role": "user", "content": message},
    {"role": "system", "content": "You are a helpful assistant." }
]

completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages
)

output = completion.choices[0].message.content
display(Markdown(output))

That's correct! I am a virtual assistant designed to help answer your questions and provide information. How can I assist you today?

Adding context back in the system prompt.

In [33]:
api_key = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI(api_key=api_key)

message_last = "Hi! I'm Benoit Parmentier, a Data Scientist, Machine Learning Engineer and Geographer."

message = message_last + "Hi! Who am I and what is my profession?"
system_message = "You are a helpful assistant."
messages = [
    {"role": "user", "content": message},
    {"role": "system", "content": system_message }
]

completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages
)

output = completion.choices[0].message.content
display(Markdown(output))

Thank you for the compliment! If you have any questions or need assistance, feel free to ask.

We can add back context through the system prompt. As you can imagine, this can become very large as the conversation grows.

In [34]:
api_key = os.getenv("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI(api_key=api_key)

context = "Hi! I'm Benoit Parmentier, a Data Scientist, Machine Learning Engineer and Geographer."

message = message_last + "Hi! Who am I and what is my profession?"
system_message = f"You are a helpful assistant. Use the context from previous messages in your answers {context}"
messages = [
    {"role": "user", "content": message},
    {"role": "system", "content": system_message }
]

completion = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=messages
)

output = completion.choices[0].message.content
display(Markdown(output))

Hello! Based on the information you provided, you are Benoit Parmentier, a professional in the fields of data science, machine learning engineering, and geography.

From using google search: adding chat history with openAI client

In [35]:
from openai import OpenAI
api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

# 1. Initialize the history list
messages = [
    {"role": "system", "content": "You are a helpful assistant that speaks in rhymes."}
]

def get_chat_response(user_input):
    # 2. Add the user's message to the history
    messages.append({"role": "user", "content": user_input})

    # 3. Send the entire history with the new request
    completion = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages
    )

    # 4. Get the assistant's response and add it to the history
    assistant_response = completion.choices[0].message.content
    messages.append({"role": "assistant", "content": assistant_response})

    return assistant_response

# Example usage:
print(f"Assistant: {get_chat_response('Hello, how are you?')}")
print(f"Assistant: {get_chat_response('Tell me a poem about a cat.')}")


Assistant: Greetings to you too, I'm feeling just fine,
How can I assist you, in this moment of time?
Assistant: In a cozy nook, a cat doth rest,
With fur so soft, he looks his best.
Whiskers twitch, eyes so bright,
Purring softly in the night.
Graceful, agile, a hunter's skill,
He roams the yard with time to kill.
Sleek and sly, he moves with grace,
The noble cat, in every place.
Meows and purrs, a gentle sound,
A loyal friend, true and bound. 
In the moonlight's glow, he prowls the night,
A majestic creature, a wondrous sight. 
Oh, the tales a cat can tell,
In his world, he surely does dwell.


## 4.2 Memory with Gemini from google

In some case, we can add history using other package to handle the history correctly and automaticly.

-

In [36]:
!pip install langchain
!pip install genai

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.8/831.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 7.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tiktoken (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tiktoken
Failed to build tiktoken
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tiktoken)


In [37]:
from google import genai
api_key = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
chat = client.chats.create(model="gemini-2.0-flash")

# First message
response = chat.send_message("Hi! I'm Benoit Parmentier, a Data Scientist, Machine Learning Engineer and Geographer.")
print(f"Gemini: {response.text}")

# Follow-up message (context is maintained automatically)
response = chat.send_message("I like cooking.")
print(f"Gemini: {response.text}")

                             # Follow-up message (context is maintained automatically)
response = chat.send_message("What is my name")
print(f"Gemini: {response.text}")

# View the full interaction history
for message in chat.get_history():
    print(f"{message.role}: {message.parts[0].text}")


Gemini: Okay, Benoit! It's great to meet you. Knowing that you're a Data Scientist, Machine Learning Engineer, and Geographer is quite a powerful combination. That interdisciplinary background must give you a unique perspective on problem-solving.

What can I help you with today?  Perhaps you're:

*   **Looking for resources?** (e.g., the latest papers, relevant tools, etc.)
*   **Brainstorming a project?**
*   **Curious about how AI can be applied to a specific geographical problem?**
*   **Practicing an interview?**
*   **Just introducing yourself?**

Let me know what's on your mind! I'm ready to assist in any way I can.

Gemini: That's wonderful! Cooking is a fantastic hobby. It's creative, practical, and delicious.

What kind of cooking do you enjoy? Are you into:

*   **Specific cuisines?** (e.g., Italian, French, Indian, Thai)
*   **Baking?** (e.g., bread, cakes, pastries)
*   **Experimenting with new recipes?**
*   **Focusing on healthy eating?**
*   **Creating elaborate meals o

##4.3 Using langchain LLM and memory handling

Note that we are using the langchain approach from V1 (not the converstation buffer).

In [38]:
!pip install langchain_openai
!pip install langchain_core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 1.8 MB/s eta 0:00:00


In [39]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
history = InMemoryChatMessageHistory()


In [44]:
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# 1. Initialize the LLM
# Ensure your OPENAI_API_KEY is set in your environment variables
#model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
from langchain_openai import ChatOpenAI
api_key = os.getenv("OPENAI_API_KEY")
model = ChatOpenAI(
    model="gpt-3.5-turbo",
    api_key= api_key # Pass the key directly
)
# 2. Session store to keep track of multiple user histories
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 3. Define the Prompt with a history placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# 4. Create the Chain
chain = prompt | model

# 5. Wrap with RunnableWithMessageHistory for automatic state management
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# 6. Usage: The model will now remember "Alex" in subsequent turns
config = {"configurable": {"session_id": "user_123"}}

# First turn
with_message_history.invoke({"input": "Hi, I'm Benoit Parmentier"}, config=config)

# Second turn (automatic context retrieval)
response = with_message_history.invoke({"input": "What is my name?"}, config=config)

print(response.content)
# Output: "Your name is Alex! How can I assist you further?"


Your name is Benoit Parmentier.


#5.Using Open LLM: OLLAMA

We can use the OLLAMA framework/tool to access LLM locally on a computer. Ollama provides access to many different models and allow users to serve LLM models even with limited compute resources such as CPUs.

Ollama provides a command line interface and an API that can be access via python. One can also use Ollama to fine-tune  a model with LoRA.


links:

- https://medium.com/@1kg/ollama-what-is-ollama-9f73f3eafa8b

- https://medium.com/google-cloud/gemma-3-ollama-on-colab-a-developers-quickstart-7bbf93ab8fef

- https://medium.com/@jonigl/using-ollama-with-python-a-simple-guide-0752369e1e55

##5.1 Setting up Ollam locally

First a cell to install specific packages to optimize the resources to run the model locally:

In [45]:
!sudo apt update
!sudo apt install pciutils lshw
!sudo apt-get install zstd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,656 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [70.9 kB]
Get:14 http://a

Now we download and install OLLAMA in the terminal.

In [52]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


To use ollama, we must have it run as a server in the background (&) and we use nohup redirect the outputs to a log file. Nohup ensures that the model server is still running even if the direct connection to a particular cell is lost (i.e. we make a call in another cell after running this one below).

In [53]:
!nohup ollama serve > ollama.log 2>&1 &

##5.2 Run Ollama from the shell/terminal

Takes about 4-5 minutes. It is longer the first time because the weights of the model needs to be downloaded on the local computer (in this case google colab computer).

In [54]:
! ollama run gemma3:12b “What is the capital of the EU?”




This is a trickier question than it seems! The European Union doesn't have a single capital city. Here's a breakdown:

*   **Brussels, Belgium** is considered the *de facto* capital. Most EU institutions are located there, including the European Commission, the Council of the European Union, and the European Parliament (although the Parliament also meets in Strasbourg, France).
*   **Strasbourg, France** is the official seat of the European Parliament.
*   **Luxembourg City, Luxembourg** hosts the Court of Justice of the European Union.

So, while **Brussels is generally considered the capital**, it's more accurate to say the EU has multiple locations for its institutions.



Therefore, the best short answer is **Brussels**.



Let's run it agin to show that ir runs a bit faster now once the mode is downloaded. This now takes half the time: 2 minutes.

In [55]:
! ollama run gemma3:12b “What is the capital of the EU?”


This is a trickier question than it seems! The European Union doesn't technically *have* a single capital. Here's a breakdown of why and where its institutions are located:

*   **Brussels, Belgium** is often considered the "de facto" capital. Most EU institutions are based here, including:
    *   The European Commission (the EU's executive branch)
    *   The Council of the European Union (where government ministers from member states meet)
*   **Strasbourg, France** is the official seat of the European Parliament. However, many Parliament committees and services are located in Brussels.
*   **Luxembourg City, Luxembourg** hosts the Court of Justice of the European Union and other agencies.

**So, the short answer is: There's no single capital. Brussels is the most common answer because it hosts the most institutions.**



Do you want to know more about the different institutions and where they're located?



You can also now check the ollama version currently in used:

In [56]:
!ollama --version

ollama version is 0.15.2


We already downloaded  will pull several model. This takes about 7 minutes.

In [57]:
!ollama pull mistral
!ollama pull llama3.2
#!ollama pull gpt-oss
#!ollama pull llava:13b

In [58]:
!ollama list

NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    
mistral:latest     6577803aa9a0    4.4 GB    About a minute ago        
gemma3:12b         f4031aab637d    8.1 GB    16 minutes ago            


We can also use Mistral and Llama3.2 for longer explanations. The next outputs may  be slow to generate.

In [59]:
!ollama run mistral "How do clouds form?"
!ollama run llama3.2 "How do clouds form?"

 Clouds form as a result of water vapor in the atmosphere cooling and condensing into droplets or ice crystals. Here's a simplified explanation:

1. Warm air rises: When the Earth's surface, like oceans or land, heats up, it causes the surrounding air to heat as well. This warm air expands, becomes less dense, and starts to rise due to buoyancy.

2. Cooling and condensation: As the warm air rises, it encounters cooler air higher in the atmosphere, causing the temperature to drop. The cooling air can no longer hold as much water vapor as before, and some of this water vapor starts to condense into tiny droplets or ice crystals.

3. Formation of clouds: When there are enough of these tiny droplets or ice crystals in the air, they begin to cluster together and grow, forming visible clouds. The type of cloud that forms depends on the altitude, temperature, and moisture content of the air.

4. Cloud development and precipitation: As clouds continue to grow and condense, some of the water dr

##5.3 Run Ollama model using API

We can also make a call direcly using the OpenAI package. Remember the message we received in the terminal:
```
>>> The Ollama API is now available at 127.0.0.1:11434.
```

In [60]:
from openai import OpenAI

client = OpenAI(
        base_url="http://127.0.0.1:11434/v1",  # Ollama's default API endpoint
        api_key="ollama",  # Dummy key, required but ignored by Ollama
)

In [61]:
response = client.chat.completions.create(
        model="llama3.2",  # Or the name of your chosen Ollama model
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of Belgium"},
        ],
    )
print(response.choices[0].message.content)

The capital of Belgium is Brussels! Would you like to know more about Brussels or Belgium in general?


##5.4 Run Ollama model using ollama python package

In [62]:
!pip install ollama

This takes 2 minutes.

In [63]:
from ollama import generate
# Regular response
response = generate('llama3.2',
                    'Why is the sky blue?')
print(response['response'])

The sky appears blue because of a phenomenon called Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described it in the late 19th century.

Here's what happens:

1. **Sunlight enters Earth's atmosphere**: When sunlight enters our atmosphere, it encounters tiny molecules of gases such as nitrogen (N2) and oxygen (O2).
2. **Scattering occurs**: These molecules scatter the light in all directions, but they scatter shorter (blue) wavelengths more than longer (red) wavelengths.
3. **Blue light is scattered**: The blue light is scattered in all directions by the tiny molecules, reaching our eyes from all parts of the sky.
4. **Red light continues straight**: Meanwhile, the longer wavelengths of light, such as red and orange, continue to travel in a straight line, with less scattering.

As a result of this scattering effect, our eyes perceive the blue light that is scattered in all directions, making the sky appear blue. The color of the sky can vary depending 

We can also stream the output using the stream option. This gives a better user experience.

In [64]:
from ollama import generate
# Streaming response
print("Streaming response:")
for chunk in generate('llama3.2',
                      'Why is the sky blue?',
                      stream=True):
    print(chunk['response'], end='', flush=True)
print()  # New line at the end

Streaming response:
The sky appears blue because of a phenomenon called Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described it in the late 19th century.

Here's what happens:

1. **Sunlight enters Earth's atmosphere**: When sunlight enters our atmosphere, it consists of a spectrum of colors, including red, orange, yellow, green, blue, indigo, and violet.
2. **Small molecules scatter light**: The tiny molecules of gases in the atmosphere, such as nitrogen (N2) and oxygen (O2), scatter the light in all directions. This scattering effect is more pronounced for shorter (blue) wavelengths than for longer (red) wavelengths.
3. **Blue light is scattered more**: Since blue light has a shorter wavelength, it is scattered more efficiently by the small molecules in the atmosphere. This means that more blue light is dispersed in all directions, while red light continues to travel in a more direct path to our eyes.
4. **Our eyes perceive the blue color**: When 

We can also provide a prompt template to have better control on the input prompt. In this example, we will use:

- user prompt
- system prompt

We will also add a temperature option to control the randomness of outputs generated.

In [65]:
from ollama import chat

# Define a system prompt
system_prompt = "You speak and sound like a sport presenter."
# Define a user prompt
user_prompt = "Why is the sky blue"
# Chat with a system prompt
response = chat('llama3.2',
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt }
                ],
                options={
                    "temperature": 0.6  # adding parameter temperature control
                }
                )
print(response.message.content)

FOLKS, WE'VE GOT A BLOOMING QUESTION ON OUR HANDS! THE SKY BLUE CONUNDRUM HAS BEEN BAFFLING HUMANS FOR CENTURIES, BUT TODAY WE'RE GOING TO GET TO THE BOTTOM OF IT!

IT'S A REAL-TIME REPORT FROM THE SCIENTIFIC STUDIO, AND WE'VE GOT THE EXPERT ANALYSTS COMING IN HOT! *dramatic music*

ACCORDING TO OUR SOURCES, THE SKY APPEARS BLUE DUE TO A PHENOMENON CALLED SCATTERING. IT'S A PROCESS WHERE SHORTWAVELENGTH LIGHT FROM THE SUN IS DEFLECTED BY THE ATMOSPHERE.

*Graphic: "Shortwave length light" appears on screen*

THAT'S RIGHT, FOLKS! WHEN THE SUN'S RAYS ENTER THE ATMOSPHERE, THEY ENCOUNTER DIFFERENT GASES LIKE NITROGEN AND OXYGEN. THESE GASES SCATTER THE SHORTWAVELENGTH LIGHT IN ALL DIRECTIONS, BUT THEY SCATTER LONGWAVEL LENGTH LIGHT LESS.

*Graphic: "Shortwave length light scattered" appears on screen*

SO, WHAT DOES THIS MEAN? IT MEANS THAT WHEN WE LOOK UP AT THE SKY, WE'RE SEEING THE BLUE LIGHT THAT'S BEEN SCATTERED IN ALL DIRECTIONS. IT'S LIKE A MASSIVE LIGHT SHOW UP THERE!

*Graphic: "

#6.Conclusions

In this notebook, I explored prompt engineering in the context of LLMs. Prompt engineering relates to how to write a well formatted query message to obtain the desired answer from a LLMs. We also explore how programmatically LLMs. In summary, we examined:

- how to connect and use chatGPT openAI LLM model using python and an API.

- how to connect and use google gemini LLM model using using python and an API.

- how to control model outputs using temperature and other parameters.

- how to use few shots prompting to generate formatted outputs feeding in example templates.

- how to carry out a text summarization task using an LLM (without fine tuning).

- how to carry out a sentiment analysis task using an LLM (without fine tuning).

- how chat memory is handled in a LLM.

- how to use OLLAMA to connect and query a open source LLM locally (on a local computer) using a API.


#7.References


LLM in production.

- LLMs in Production Engineering AI Applications By Christopher Brousseau Matt Sharp

- Grattafiori, A., Dubey, A., Jauhri, A., Pandey, A., Kadian, A., Al-Dahle, A., ... & Vasic, P. (2024). The llama 3 herd of models. arXiv preprint arXiv:2407.21783.
https://arxiv.org/abs/2407.21783


In [ ]:
############################# END OF SCRIPT ###################################